# Chicago Crime Composition Analysis (2001–2026)

> *Did COVID actually break Chicago's crime patterns, or did the changes that look like a COVID shock start years earlier?*

## The Question

In 2020 and 2021, almost every news story about urban crime treated the pandemic as a turning point, and many cities responded by reorganizing policing budgets and policy frameworks around the assumption that COVID-19 caused a discrete break in how crime occurred. This project tests that assumption directly: using 25 years of Chicago crime data (2001–2026, $\approx$8.5 million reported incidents across 25 FBI categories), I ask whether the *composition* of crime, the mix of crime types reported each month, actually broke at the pandemic, or whether the post-2020 changes that look like a shock were already in motion before COVID arrived.

## The Answer

**The data does not support COVID as a structural break in Chicago's crime composition.** Four independent statistical methods agree that the composition evolved gradually rather than in discrete regimes. Multivariate changepoint detection finds no breaks at pandemic boundaries under conventional evidence thresholds. The largest post-2020 compositional shifts trace to the multi-year decline of drug-enforcement categories, a trend that began before 2010 and continued through the pandemic rather than to the pandemic itself.

## Why This Matters

Policy responses to crime composition implicitly assume a model of what's happening: cities that increased violent-crime budgets in 2020, based on a "COVID shock" framing, were responding to the *tail* of a multi-year trend, not to a discrete pandemic-caused break. If the underlying pattern is gradual drift, there is no pre-pandemic equilibrium for the system to revert to, meaning the policy framework needs to account for accumulated change rather than acute disruption. This distinction matters for any organization (city government, research nonprofit, journalist, or analyst) that interprets post-pandemic urban crime statistics.

## Approach

The analysis treats monthly crime composition as a multivariate non-stationary system, transformed via the centered log-ratio (CLR) function to put compositional data into a space where standard statistical methods apply correctly. Four lines of evidence test whether the pandemic acted as a structural break:

1. **Stationarity testing** - Does each crime category's share have a consistent statistical structure over time?
2. **Block bootstrap** - Are average compositions in the pre-COVID, COVID, and post-COVID eras statistically distinguishable?
3. **Multivariate changepoint detection** - letting the algorithm tell us where structural breaks occur, without assuming where they should be
4. **Dependence-structure comparison** - has the *correlation between* crime categories shifted across eras, beyond just the average shares?

## Era Definitions

Three administrative reference windows are used throughout for descriptive comparison. **These are policy-defined boundaries; Section 6c documents that data-driven changepoint detection does not identify these dates as empirical regime boundaries.**

| Era | Period | Months |
|---|---|---|
| Pre-COVID | January 2001 – February 2020 | 230 |
| COVID | March 2020 – December 2022 | 34 |
| Post-COVID | January 2023 – April 2026 | 40 |

Era cutoffs are anchored to the WHO pandemic declaration (March 2020) and the expiration of Illinois's disaster proclamation (late 2022). The underlying behavioral and enforcement shifts may not align exactly with these dates, which is part of what this analysis tests.

---

*Author: Erik Pak | May 2026 | Data: [Chicago Data Portal - Crimes 2001 to Present](https://data.cityofchicago.org/Public-Safety/Crimes-2001-to-Present/ijzp-q8t2)*

## Table of Contents

**Section 1 - Data**
- [1.1 Data Import](#1.1-Data-Import)
- [1.2 Data Integration & Cleaning](#1.2-Data-Integration-&-Cleaning)
- [1.3 Full Diagnostic](#1.3-Full-Diagnostic)
- [1.4 Missingness Check](#1.4-Missingness-Check)
- [1.5 Onset Diagnostic (Involuntary Manslaughter)](#1.5-Onset-Diagnostic)

**Section 2 - User Functions / Remove/ Fill Missing Data & Report**
- [2.1 User Function(s)](#2.1-User-Functions)
- [2.2 Remove Feature](#2.2-Remove)
- [2.3 Fill Missing Data & Report](#2.3-Fill-Report)
- [2.4 Validate Filled Data](#2.4-Validate)

**Section 3 - CLR Transformation & Epsilon Selection**
- [3.1 CLR Transformation](#3.1-CLR-Transformation)
- [3.2 Epsilon Grid Sweep](#3.2-Epsilon-Grid-Sweep)
- [3.3 Initial PCA on Chosen Epsilon](#3.3-Initial-PCA)

**Section 4 - Sparse-Category Deep Dives**
- [4.1 Gambling](#4.1-Gambling)
- [4.2 Prostitution](#4.2-Prostitution)

**Section 5 - Three-Way PCA Sensitivity (A / B / C)**
- [5.1 Scaled vs Unscaled Rationale](#5.1-Scaled-vs-Unscaled)
- [5.2 Three-Way PCA Comparison](#5.2-Three-Way-PCA)
- [5.3 PC Score Trajectories](#5.3-PC-Score-Trajectories)
- [5.4 Aitchison Plot & PC Loading Stability](#5.4-Aitchison-Plot)
- [5.5 Epsilon Stability Diagnostic](#5.5-Epsilon-Stability)

**Section 6 - Per-Era Distribution & Era Comparison**
- [6.1 Slice CLR into Eras](#6.1-Slice-Into-Eras)
- [6.2 Per-Era Distributional Parameters](#6.2-Per-Era-Parameters)
- [6.3 Covariance Sanity Check](#6.3-Covariance-Sanity-Check)
- [6.4 Mean & Volatility Heatmaps](#6.4-Heatmaps)
- [6.5 Per-Category CLR Distribution Audit](#6.5-Distribution-Audit)

**Section 6 (cont.) - Stationarity & Bootstrap**
- [6a Stationarity Testing (ADF / KPSS / Zivot-Andrews)](#6a-Stationarity)
- [6b Block Bootstrap Test (Pre-COVID / COVID / Post-COVID)](#6b-Bootstrap)

**Section 6c - Multivariate Changepoint Detection**
- [6c.1 Confirmatory: Dynp at n_bkps=2](#6c.1-Dynp-Confirmatory)
- [6c.2 Sensitivity: Vice-Included vs Vice-Excluded](#6c.2-Sensitivity)
- [6c.3 Pelt Exploratory Penalty Sweep](#6c.3-Pelt)
- [6c.4 PC1 Trajectory Visualization](#6c.4-Visualization)

**Section 7 - Per-Era Co-Movement Structure**
- [7a Frobenius Distance Between Era Correlation Matrices](#7a-Frobenius)
- [7b Crime-Pair Correlation Shifts (Element-Wise)](#7b-Element-Wise)

## Importing Libraries

In [ ]:
import os
import pandas as pd
import pyarrow as pa
import pyarrow.feather as feather
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
from sklearn.preprocessing import StandardScaler
import ruptures as rpt
from sklearn.covariance import LedoitWolf

# Path + custom modules
sys.path.append('../modules/') 
import clr_config as cfg
import clr_utilities as utils
import clr_eps_grid as grid
import clr_analyze_plot as plot
import clr_era as clr
import clr_pca_sign_normalization as pca_anchor

# Image Path
image_path = "../figures/"

# Seed
_RANDOM_STATE = 1776

# Library versions
versions = {
  "Python" : sys.version.split()[0],
  "Pandas" : pd.__version__,
  "NumPy"  : np.__version__,
  "Pyarrow" : pa.__version__,
  "Seaborn" : sns.__version__,
  "Matplot" : sys.modules['matplotlib'].__version__,
  "Ruptures": sys.modules['ruptures'].__version__,
  "Scikit-learn"  : sys.modules['sklearn'].__version__,
  "Statsmodels" : sys.modules['statsmodels'].__version__,
  "Scipy"  : sys.modules['scipy'].__version__,
}
df_versions = pd.DataFrame(list(versions.items()), columns=['Library', 'Version'])
print(df_versions)
# Remove scientific notation
np.set_printoptions(suppress=True, precision=6, linewidth=100)

✓ Module path found: /Users/sir/Desktop/Project/ChicagoCrimeCompositionAnalysis/modules
        Library  Version
0        Python   3.13.9
1        Pandas    3.0.3
2         NumPy    2.4.6
3       Pyarrow   24.0.0
4       Seaborn   0.13.2
5       Matplot   3.10.9
6      Ruptures  v1.1.10
7  Scikit-learn    1.8.0
8   Statsmodels   0.14.6
9         Scipy   1.17.1


## <a id="1.1-Data-Import"></a> 1. Data Import

In [ ]:
# PyArrow's version of the 'arrow' backend
df = feather.read_feather('../Data/crime_data.feather', memory_map=True, types_mapper=pd.ArrowDtype)
# display
df.head()

### <a id="1.2-Data-Integration-&-Cleaning"></a> Data Integration & Cleaning

In [ ]:
# New column
df['year_month'] = df['date'].dt.strftime('%Y%m')
# Normalize era keys to consistent labels
df['era'] = (
    df['era']
    .map(cfg.era_map)
    .fillna(df['era'])
    .astype(str)                 # ensure all values are strings
    .astype('string[pyarrow]')   # Arrow conversion
)
# display
df.era.unique()

In [ ]:
# get the dates of period in the dataset
start_date = pd.to_datetime(df[cfg.config_agg["_DATE_KEY"]].min(), format='%Y%m').date().strftime('%Y-%m')
end_date = pd.to_datetime(df[cfg.config_agg["_DATE_KEY"]].max(), format='%Y%m').date().strftime('%Y-%m') 
# print the date range
print(f"Dataset covers from {start_date} to {end_date}")

### <a id="1.3-Full-Diagnostic"></a> Full Diagnostic 

In [ ]:
# Normalize year_month to datetime (keep both versions)
df['year_month_dt'] = pd.to_datetime(df['year_month'], format='%Y%m')

# Identity check
identical = bool((df['primary_description'] == df['primary_type']).all())
print(f"primary_description == primary_type: {identical}\n")

# Distinct primary_descriptions
prim_list = sorted([str(x) for x in df['primary_description'].dropna().unique()])
print(f"Distinct primary_descriptions: {len(prim_list)}")
for c in prim_list:
    print(f"  - {c}")
print()

# Distinct FBI categories
fbi_list = sorted([str(x) for x in df['fbi_code_desc'].dropna().unique()])
print(f"Distinct fbi_code_desc: {len(fbi_list)}")
for c in fbi_list:
    print(f"  - {c}")
print()

# Era boundaries are actually encoded
era_bounds = (df.groupby('era')['year_month_dt']
              .agg(['min', 'max', 'nunique'])
              .rename(columns={'nunique': 'n_months'}))
print("Era boundaries:")
print(era_bounds.to_string())
print()

# index_code vs fbi_index_code
print("index_code vs fbi_index_code:")
print(pd.crosstab(df['index_code'], df['fbi_index_code'], dropna=False))

### <a id="1.4-Missingness-Check"></a> Check Missingness

In [ ]:
# Check integrity and missingness
missingness_results = utils.run_integrity_report(df)

### <a id="1.5-Onset-Diagnostic"></a>  Onset Diagnostic ('Involuntary Manslaughter / Reckless Homicide)

In [ ]:
# Crime type to analyze
sus = 'Involuntary Manslaughter / Reckless Homicide'

# Build a monthly time series of counts for this crime type
s = (df[df['fbi_code_desc'] == sus]      # filter to the selected crime
     .groupby('year_month_dt')           # group by month
     .size()                             # count incidents per month
     .sort_index())                      # ensure chronological order

# Basic non-zero occurrence diagnostics
print(f"First non-zero month: {s[s > 0].index.min():%Y-%m}")  # earliest month with >0 incidents
print(f"Last non-zero month:  {s[s > 0].index.max():%Y-%m}")  # latest month with >0 incidents
print(f"Total non-zero months: {(s > 0).sum()}")              # number of months with any incidents

# Summary statistics for the monthly counts
print(f"\nMonthly count statistics:")
print(s.describe())                      # mean, std, min, quartiles, max

# Yearly aggregation: how this crime behaves year by year
yearly = pd.DataFrame({
    'months_with_data': s.groupby(s.index.year).apply(lambda x: (x > 0).sum()),  # months with > 0 incidents
    'total_count':      s.groupby(s.index.year).sum(),                           # total incidents per year
    'max_in_year':      s.groupby(s.index.year).max(),                           # worst month in each year
})

# Display the yearly pattern in a readable table
print(f"\nYearly pattern:")
print(yearly.to_string())

#### Confirm

In [ ]:
iucr_by_year = (df[df['fbi_code_desc'] == sus]
                .assign(yr=lambda d: d['date'].dt.year)
                .groupby(['yr', 'iucr'])
                .size()
                .reset_index(name='n'))
print(iucr_by_year.to_string(index=False))

# Also: what IUCR codes appear in this FBI category overall?
codes = df.loc[df['fbi_code_desc'] == sus, 'iucr'].value_counts()
print(f"\nIUCR codes mapped to this FBI category:")
print(codes)

#### 📌 Summary
> - Involuntary Manslaughter / Reckless Homicide (IUCR codes 0141 and 0142) was excluded from the compositional analysis because the category was too sparse for reliable modeling. Across 304 months of data, it appeared in only 23.4% of months and totaled just 89 incidents overall, averaging fewer than one incident every three months (0.29 per month). Including such a low-frequency category in a high-dimensional compositional framework could introduce instability and distort the overall covariance structure. As a result, the final compositional analysis was conducted using the remaining 25 FBI crime categories, all of which showed at least 90.8% monthly coverage.

> - Although excluded from formal modeling, the category still showed an observable increase after 2017. Between 2018 and 2023, Chicago recorded an average of roughly 7 incidents per year (ranging from 6 to 12), compared with about 2 incidents per year (ranging from 1 to 4) between 2011 and 2017. While the counts remain too small for dependable statistical inference, the increase may reflect changes in prosecutorial practices related to drug-induced homicide cases or shifts in how fatal traffic incidents were classified during that period.

## <a id="2.1-User-Functions"></a> 2. User Function(s)

In [ ]:
def pca_prep_comparison(
    epsilon_sweep_result: dict,
    epsilon: float,
    epsilon_alt: float
):

    # Build comparison list
    eps_list = [epsilon, epsilon_alt]

    # Storage for results
    results_pca = {}

    # Loop over epsilons and generate plots
    for eps in eps_list:
        clr_df = epsilon_sweep_result["clr_dict"][eps]

        result = pca_anchor.compute_pca(
            clr_df,
            epsilon=eps,
            verbose=False,
        )

        results_pca[eps] = result

    return results_pca



def check_clr_health(sweep_result, target_eps=None):
    """
    Validates the numerical structure and CLR zero‑sum health for a specific epsilon.

    Args:
        sweep_result (dict): Output from sweep_epsilon_grid(...).
        target_eps (float, optional): Epsilon to inspect. Defaults to chosen_eps.

    Returns:
        dict: The full clr_dict (unchanged), or None if epsilon is missing.
    """

    # Default to auto‑chosen epsilon
    if target_eps is None:
        target_eps = sweep_result['meta']['chosen_eps']

    clr_dict = sweep_result.get('clr_dict', {})

    # Validate epsilon exists
    if target_eps not in clr_dict:
        print(f"⚠️ Warning: Epsilon {target_eps:.10f} not found in clr_dict.")
        return None

    # Extract the CLR dataframe
    target_df = clr_dict[target_eps]
    rows, cols = target_df.shape

    # Temporal range formatting
    start_dt = target_df.index[0].strftime('%Y-%m')
    end_dt   = target_df.index[-1].strftime('%Y-%m')

    # CLR zero‑sum health check
    mean_abs_sum = target_df.sum(axis=1).abs().mean()
    health_status = "🟢 OPTIMAL" if mean_abs_sum < 1e-12 else "🔴 RE‑CENTERING REQUIRED"

    # Output summary
    print(f"{' ERA STRUCTURE ':=^50}")
    print(f"Container Type    : {type(clr_dict).__name__}")
    print(f"Total Categories  : {cols}")

    print(f"\n{' TARGET EPSILON: ' + f'{target_eps:.10f}':-^50}")
    print(f"Dimensions        : {rows} months x {cols} categories")
    print(f"Temporal Range    : {start_dt} to {end_dt}")
    print(f"Primary Features  : {', '.join(target_df.columns[:3])} ...")
    print(f"Numerical Health  : {mean_abs_sum:.2e} ({health_status})")
    print(f"{'':=^50}")

    return clr_dict

### <a id="2.2-Remove"></a> Remove Involuntary Manslaughter / Reckless Homicide

In [ ]:
# Remove Involuntary Manslaughter / Reckless Homicide
df_new = df[df.fbi_code_desc != sus]
# shape
print(f"Total Rows: {(df.shape[0]):,} & New Total Rows: {(df_new.shape[0]):,}")

### <a id="2.3-Fill-Report"></a> Fill Missing Data & Report

In [ ]:
# Fill missing data and report
fill_results = utils.fill_missing(df_new)

### <a id="2.4-Validate"></a> Validate Filled Data

In [ ]:
# Validate filled data
utils.validate_crime_data(fill_results['filled_df'], zero_rate_threshold = 0.05)

## <a id="3.1-CLR-Transformation"></a> 3. Centered Log-Ratio (CLR) Transformation
> ❗ Note: Centered Log-Ratio transformation (CLR) measures the relative share of each crime category within the overall crime composition, not the raw number of crimes.
#### CLR Formula:
- $CLR(x_i) = ln(x_i + \varepsilon) - \frac{1}{K} \sum\limits_{j=1}^{K} ln(x_j + \varepsilon)$
  - $x_{i}$: The $i$-th component (or part) of the composition
  - $K$: The total number of parts (components) in the composition
  - $\frac{1}{K} \sum\limits_{j=1}^{K}ln(x_j + \varepsilon)$: This represents the arithmetic mean of the natural log of all components. This is equivalent to the logarithm of the geometric mean of the components, which is the standard reference point for the CLR
  - $\varepsilon$ (Pseudocount): A small value (often 0.5, 1, or a small constant) added to components, primarily to handle zero values in the data, as $ln(0)$ is undefined.

### <a id="3.2-Epsilon-Grid-Sweep"></a> Epsilon Grid Sweep

In [ ]:
# Find delta constant
epsilon_result = grid.build_eps_grid(fill_results['filled_df'])
# Select Optium esp
epsilon_sweep_result = grid.sweep_epsilon_grid(epsilon_result['pivot_data'], epsilon_result['eps_values'], auto_select=True, plot=True)

In [ ]:
# Extracts the transition window 
df_view, chosen_eps, prior_eps = utils.get_epsilon_transition(epsilon_sweep_result, window_prior=5, window_past=5)
df_view

### <a id="3.3-Initial-PCA"></a> Initial PCA on Chosen Epsilon

In [ ]:
# PCA for CLR: Sparse
pca_result = pca_anchor.compute_pca(epsilon_sweep_result['clr_dict'][chosen_eps], epsilon=chosen_eps)

In [ ]:
# Plot PCA
three_loadings_result = pca_anchor.plot_three_loadings(pca_result, epsilon=chosen_eps)

## 4. Sparse-Category Deep Dives

### <a id="4.1-Gambling"></a> Gambling

In [ ]:
# Copy the aggregated, filled monthly crime data
data = fill_results['filled_df']

# Extract the monthly time series for the crime category "Gambling"
gambling = (data[data['fbi_code_desc'] == 'Gambling']   # filter to Gambling incidents
            .set_index('year_month')['crime_count']     # use year_month as index, select counts
            .sort_index())                              # ensure chronological order

# Basic counts of how often Gambling appears
print(f"Total months for Gambling: {len(gambling)}")          # total months in the series
print(f"Zero months: {(gambling == 0).sum()}")                # months with zero incidents
print(f"Non-zero months: {(gambling > 0).sum()}\n")           # months with >0 incidents

# Identify all months where the gambling count is zero
gambling_zeros = gambling[gambling == 0].index

# Count zero months by calendar year
print(f"Zero months by year:")
zeros_by_year = gambling_zeros.to_series().dt.year.value_counts().sort_index()
print(zeros_by_year.to_string())

# Print every zero month in chronological order
print(f"\nAll zero months (chronologically):")
for d in gambling_zeros:
    print(f"  {d.strftime('%Y-%m')}")

# Yearly summary statistics for Gambling incidents
print(f"\nGambling yearly totals (full series):")
yearly = gambling.groupby(gambling.index.year).agg(['sum', 'mean', 'min', 'max'])
yearly.columns = ['total', 'mean_per_month', 'min', 'max']    # rename columns for clarity
print(yearly.to_string())

In [ ]:
# Confirm the 99% claim directly
peak_window = gambling.loc['2006-01-01':'2008-12-01'].mean()
trough_window = gambling.loc['2020-01-01':'2025-12-01'].mean()
decline = 1 - (trough_window / peak_window)

print(f"Mean monthly count, 2006–2008 (peak window): {peak_window:.1f}")
print(f"Mean monthly count, 2020–2025 (trough window): {trough_window:.2f}")
print(f"Decline from peak window to trough window: {decline:.1%}")

| Period | Mean per month | Approximate decline |
|---|---:|---:|
| 2001–2008 (peak era) | 81–118 | Baseline |
| 2009–2012 | 60–83 | Modest decline |
| 2013–2015 | 26–50 | Substantial decline |
| 2016–2019 | 12–17 | Near-collapse |
| 2020 onward | 0.25–2.2 | $\approx$99% reduction from peak |


- 2009: Legislation Passed
    - July 13, 2009: Governor Pat Quinn signed the Video Gaming Act into law. It was introduced to fund a $31 billion state infrastructure improvement plan. The law authorized up to 5 VGTs in licensed retail establishments, truck stops, and veteran/fraternal organizations.
    - October 19, 2009: The Illinois Gaming Board (IGB) issued the initial emergency rules to establish the regulatory framework and licensing processes.
- 2012: VGTs Go Live
    - September 10, 2012: The Central Communications System officially launched, and the first legal VGTs went live for patron play across the state.
- 2019: Major Expansion
    - June 2019: The Illinois Gambling Expansion Bill (Public Act 101-0031) was signed into law. This legislation significantly expanded the Video Gaming Act by:
        - Increasing the maximum number of VGTs allowed in licensed truck stops from 5 to 10.
        - Increasing the maximum number of VGTs allowed in fraternal and veterans establishments from 5 to 10.
        - Raising the maximum tax rate on video gaming net terminal income.
- 2021: Post-Pandemic Adjustments.
    - October 2021: House Bill 3136 was passed, implementing further structural and regulatory updates across the state's gaming laws. This bill refined operational rules for VGTs, licenses, and revenue sharing.
- 2024–2026: Modernization and Regulatory Updates
    - 2024–2025: The IGB continued to propose and draft rule updates regarding cashless wagering, Self-Exclusion Programs, and Sales Agent Licensure to modernize oversight.
    - Ongoing: The network has grown to nearly 50,000 VGTs statewide. However, municipalities (most notably Chicago) retain the local option to prohibit video gaming terminals within their city limits.


**Note:**
> Monthly Gambling counts declined nearly monotonically over the panel, from a peak of 118.8 incidents per month in 2007 to 11.9 in 2019, then a further sharp drop to 1–2 per month from 2020 onward, a 98.8% decline between the peak window (2006-2008, mean 112.4/month) and the trough window (2020-2025, mean 1.4/month). The decline began before the July 13, 2009, enactment of the Illinois Video Gaming Act and continued through the rollout of legal Video Gaming Terminals (first patron play September 10, 2012) and the 2019 Gambling Expansion Bill. Critically, Chicago itself maintained a municipal ban on VGTs throughout the panel period (lifted only in late 2025), so the decline cannot be attributed to legal in-city substitutes for illegal gambling. Plausible contributors include geographic displacement of demand to legal suburban venues, secular de-prioritization of vice enforcement by CPD, and shifts of illegal gambling toward online platforms. Gambling remains in the compositional analysis, but post-2015 CLR values are influenced by the choice of pseudocount and should be interpreted with this in mind.

### <a id="4.2-Prostitution"></a> Prostitution

In [ ]:
# Extract monthly time series for Prostitution incidents
prost = (data[data['fbi_code_desc'] == 'Prostitution']   # filter to Prostitution cases
         .set_index('year_month')['crime_count']          # use year_month as index, select counts
         .sort_index())                                   # ensure chronological order

# Yearly summary statistics for Prostitution
yearly = prost.groupby(prost.index.year).agg(['sum', 'mean', 'min', 'max'])
yearly.columns = ['total', 'mean_per_month', 'min', 'max']  # rename for clarity
print("Prostitution yearly pattern:")
print(yearly.to_string())

# Compare historical high-activity period vs recent low-activity period
peak = prost.loc['2001-01-01':'2008-12-01'].mean()          # average monthly incidents in early 2000s
trough = prost.loc['2020-01-01':'2025-12-01'].mean()        # average monthly incidents in recent years

print(f"\nPeak window mean (2001–2008): {peak:.1f}/month")
print(f"Trough window mean (2020–2025): {trough:.2f}/month")

# Compute proportional decline if peak is non-zero
if peak > 0:
    print(f"Decline: {(1 - trough/peak):.1%}")

# Identify months with zero reported Prostitution incidents
zeros = prost[prost == 0]
if len(zeros) > 0:
    print(f"\nZero months ({len(zeros)} total):")
    for d in zeros.index:
        print(f"  {d.strftime('%Y-%m')}")


**Note:**
- Phase 1: Steady street economy (2001–2008). Arrests averaged 400–630/month. This was the pre-Internet street-prostitution era, when transactions happened in person, in known geographic areas, and police could make arrests through routine patrols and stings.
- Phase 2: The internet took over (2009–2015). Arrests fell from 525/month to $\approx$110/month, roughly an 80% drop. The most likely explanation is that sex work moved online, and online platforms made street solicitation unnecessary. You don't get arrested for a transaction your buyer arranged online, and that happens behind closed doors. The activity didn't disappear, but the visible, arrestable version of it did.
- Phase 3: Legacy enforcement (2016–2019). Arrests settled into $\approx$60/month. This was a new low baseline. Most sex work was online; what remained on the street was either people who couldn't or wouldn't use online platforms, or sting operations against the people still working visibly.
- Phase 4: COVID floor (2020–2025). Arrests collapsed to $\approx$10–25/month. Lockdowns shut down what street activity remained, indoor venues closed, and post-pandemic, the numbers never returned to even the 2019 baseline.

**Legally**
- A few things, none of which is the main driver of the count decline:
    - 2010 - Illinois made minors under 18 immune from prostitution prosecution
    - 2013 - Illinois made prostitution always a misdemeanor (no more felony charges)
    - 2018 - FBI seized Backpage; federal FOSTA-SESTA passed (April 2018)

**The 96% decline in prostitution arrests in Chicago isn't a 96% decline in sex work, but it's a 96% decline in the kind of sex work that's visible to police patrols.**

#### Scaled version as a sensitivity check
Our PC1 loadings (0.79 Gambling, 0.37 Prostitution, then everything else < 0.15) reflect the fact that Gambling has enormous CLR variance; its share collapsed from $\approx$0.5% to $\approx$0.005% of monthly crime over the panel, which is a log-ratio swing of roughly log(100) $\approx$ 4.6 units. Almost no other category moved that much in CLR space. So when we fit PCA on unstandardized CLR, PC1 is essentially "the Gambling axis." That's mathematically correct and substantively defensible, but Gambling really is the category with the most extreme compositional shift. But it means PC1 isn't really a multivariate finding; instead, it's a univariate finding about Gambling, dressed up as a principal component. If you z-score the CLR coordinates first, we would be asking a different question: "What patterns of co-movement exist among the categories, treating each as equally important?" That's a legitimate question too, and the resulting PCs would tell a different story, likely one with less Gambling/Prostitution dominance and more visible signal from the high-volume violent crime categories.

#### Scale Vs. Not
| Situation | Recommendation |
|---|---|
| Following Aitchison's framework strictly | Don't scale. Use CLR + un-standardized PCA. This is the textbook choice. |
| Variance is dominated by 1-2 outlier categories | Run both, compare. We are in this situation. |
| Our downstream analysis is geometric (Aitchison distance, CLR-based tests) | Don't scale. Preserves the geometry. |
| Our goal is "find co-movement clusters," treating each category as equally interesting | Consider scaling, but be aware you've departed from compositional convention. |
| Our goal is to "characterize the largest compositional shifts." | Don't scale. Unstandardized correctly identifies the biggest movers. |

##### Three Versions
| Version | Setup | Purpose |
|---|---|---|
| A - Main analysis | Unstandardized CLR (current code) | Faithful to Aitchison framework |
| B - Scaled sensitivity | Z-scored CLR coordinates | Tests whether findings depend on Gambling/Prostitution dominance |
| C - Excluding vice | Unstandardized CLR, 23 categories (drop Gambling, Prostitution) | Tests robustness to category inclusion |

## 5. Three-Way PCA Sensitivity (A / B / C)

### <a id="5.1-Scaled-vs-Unscaled"></a> Scaled vs Unscaled Rationale

In [ ]:
# Version A: un-standardized CLR (your current approach)
pca_A = pca_anchor.compute_pca(epsilon_sweep_result['clr_dict'][chosen_eps], epsilon=chosen_eps)

# Version B: z-scored CLR
scaler = StandardScaler()
clr_scaled = pd.DataFrame(
    scaler.fit_transform(epsilon_sweep_result['clr_dict'][chosen_eps]),
    columns=epsilon_sweep_result['clr_dict'][chosen_eps].columns,
    index=epsilon_sweep_result['clr_dict'][chosen_eps].index
)
pca_B = pca_anchor.compute_pca(clr_scaled, epsilon=chosen_eps)

# Version C: un-standardized CLR, excluding vice categories
exclude = ['Gambling', 'Prostitution']
clr_matrix = grid.clr_transform(counts=fill_results['filled_df'], epsilon=chosen_eps, exclude_features=exclude, verbose=True)
pca_C = pca_anchor.compute_pca(clr_matrix, epsilon=chosen_eps)

### <a id="5.2-Three-Way-PCA"></a> Three-Way PCA Comparison

In [ ]:
# Compare variance explained
# Create headers and data format templates
row_fmt = "  {:<16} | {:>8} | {:>18}"
divider = "-" * 50

print(f"\n{ '=== PCA VARIANCE EXPLAINED COMPARISON ===' :^50}")
print(row_fmt.format("Model Version", "PC1 Var", "PC1-3 Cumulative"))
print(divider)

# Version A
print(row_fmt.format(
    "A (Unscaled, 25)", 
    f"{pca_A['variance_ratio'][0]:.1%}", 
    f"{pca_A['variance_ratio'][:3].sum():.1%}"
))

# Version B
print(row_fmt.format(
    "B (Z-scored, 25)", 
    f"{pca_B['variance_ratio'][0]:.1%}", 
    f"{pca_B['variance_ratio'][:3].sum():.1%}"
))

# Version C
print(row_fmt.format(
    "C (Unscaled, 23)", 
    f"{pca_C['variance_ratio'][0]:.1%}", 
    f"{pca_C['variance_ratio'][:3].sum():.1%}"
))
print("=" * 50)

#### Three-Way PCA Comparison:

**Setup.** PCA on CLR-transformed monthly Chicago crime composition, run three ways to test whether the dominant variance structure depends on the two declining vice categories (Gambling, Prostitution).

| Version | Specification | PC1 | PC1–3 |
|---|---|---|---|
| A | Un-scaled CLR, 25 categories | 65.3% | 87.3% |
| B | Z-scored CLR, 25 categories | 56.7% | 72.2% |
| C | Un-scaled CLR, 23 categories (no vice) | 47.1% | 83.0% |

**Finding 1 - Vice contributes about a third of the leading axis.** Removing Gambling and Prostitution drops PC1 from 65.3% to 47.1%, an $\approx$18-point reduction. The two vice categories are major contributors to the dominant compositional axis.

**Finding 2 - But the co-movement is not an artifact of vice.** PC1 remains dominant at 47.1% even without the vice categories. A strong co-movement structure exists among the other 23 categories independently.

**Finding 3 - Nor is it a variance-magnitude artifact.** Z-scoring (which removes the magnitude advantage of high-variance categories) only reduces PC1 to 56.7% and is less than removing vice. If the dominance were purely a magnitude effect, z-scoring would have collapsed it, and it didn't, so the co-movement is genuine.

**Finding 4 - The headline variance number is specification-dependent.** "PC1 explains X% of variance" is 65% with vice, 47% without. Both are correct while answering slightly different questions. 

> - For "which crimes move together": Chicago crime categories exhibit strong co-movement (PC1 explains 47–65% of the variance across specifications). Gambling and Prostitution co-move most strongly and drive roughly a third of the leading axis, but a dominant co-movement structure persists among the remaining categories.
> - > The 23-cat `max |CLR|` is **10.12**, slightly *higher* than your 25-cat sweep value of $\approx$9.97. This was expected, but removing the two vice categories changed the geometric-mean denominator, which shifted every remaining CLR coordinate. It's small and harmless, but worth knowing: **our $\varepsilon = 0.00894$ was optimized for the 25-cat panel, and the 23-cat panel runs slightly higher (10.12 vs 9.97)** while holding $\varepsilon$ fixed across both analyses for comparability, and confirmed the sub-composition remained numerically stable (max|CLR| = 10.12).



### <a id="5.3-PC-Score-Trajectories"></a> PC Score Trajectories

In [ ]:
result_A_plot = plot.plot_pc_scores_over_time(pca_dict=pca_A,
                         panel_index=pca_A['observation_index'], 
                         title_suffix=' - Un-standardized CLR')

In [ ]:
result_C_plot = plot.plot_pc_scores_over_time(pca_dict=pca_C,
                         panel_index=pca_A['observation_index'], 
                         title_suffix=' - excluded [Gambling & Prostitution] CLR')

#### 📌 Summary
> Across both versions of the dataset with and without vice-related categories, the principal components reveal a consistent pattern of structural change in Chicago's crime composition over time. In both analyses, PC1 exhibits a gradual pre-COVID decline followed by a visually pronounced break at the onset of the COVID era around March 2020, a pattern consistent with a structural discontinuity rather than a continuation of prior trends (formal changepoint testing is reported below). In the vice-removed analysis, PC3 provides a distinct COVID-aligned signal: relatively stable or declining before 2020, rising during the pandemic period, then partially reverting. Its dominant loadings on Stolen Property, Motor Vehicle Theft, and Weapons Violations point to a shift in the relative weight of these property and weapons offenses during the COVID era. The post-COVID period also shows evidence of lasting compositional change: in the vice-removed analysis, PC2 rises substantially from around 2022 and remains elevated, indeed still rising through the end of the series, indicating that the composition has not reverted to its pre-COVID structure. Together, the trajectories suggest three distinguishable compositional phases: a long-term pre-COVID decline, a sharp COVID-era disruption, and a post-COVID state that settles at a new compositional configuration rather than returning to historical patterns. Because the analysis is based on CLR-transformed data, the principal component scores reflect changes in relative compositional structure rather than changes in absolute crime levels.

### Epsilon Stability Diagnostic

In [ ]:
# Iniutilize Alternative epsilon
epsilon_compare = [chosen_eps, prior_eps]
# Initialize Dictionary for outputs keyed by epsilon
results_pca = {}

# Loop over selected epsilon values and collect PC‑loading plots
for eps in epsilon_compare:

    # Generate the three‑PC loading plots for this epsilon
    r = pca_anchor.compute_pca(
        epsilon_sweep_result['clr_dict'][eps],
        epsilon=eps, verbose=True
    )

    # Store the result keyed by epsilon
    results_pca[eps] = r

#### 📌 Summary

> For the dense subset, $\varepsilon = 0.008936$ was chosen as the smallest pseudocount that removed excessively large CLR values while still preserving rank stability. To test whether this choice was fragile, we compared it with the next lower value in the grid ($\varepsilon = 0.008117$). The PCA results were essentially identical across the boundary: PC1–PC3 matched almost perfectly (cosine similarity $\approx 1.0$), and the shared subspace rotated by only $0.29^\circ$. This shows that the analysis is stable and does not depend on the exact pseudocount selected near this threshold.

### <a id="5.4-Aitchison-Plot"></a> Aitchison Plot & PC Loading Stability

In [ ]:
# Aitchison plot for the chosen epsilon (sweep_epsilon_grid() output dictionary:['clr_dict'])
aitchison_results = plot.plot_aitchison(epsilon_sweep_result['clr_dict'], chosen_eps)

### <a id="5.5-Epsilon-Stability"></a> Epsilon Stability Diagnostic

In [ ]:
# PC Loading Stability Plot for the chosen epsilon for all
pc1_stability_results = plot.plot_pc1_loading_stability(epsilon_sweep_result['clr_dict'], chosen_eps=chosen_eps, top_k=10)

#### 📌 Summary

> Both PCA plots support the same overall conclusion and show that the selected pseudocount value ($\varepsilon = 0.008938$) produces meaningful and reliable results. The analysis shows that vice-related crimes-especially Gambling and Prostitution are the main drivers behind the first principal component (PC1), which explains about 65% of the overall variation in the crime composition data. When those vice categories are removed, the influence of PC1 drops noticeably to about 47%, confirming that they strongly shape the original pattern. The chosen $\varepsilon$ value is reasonable and avoids unstable or distorted behavior. One important note is that the exact percentages and category weights shift somewhat as $\varepsilon$ changes, so the specific numbers should be viewed as approximate rather than exact. Even so, the overall findings remain consistent: vice-related offenses dominate the original compositional structure. The vice-excluded specification serves as a robustness check, confirming that substantial co-movement structure persists even in the absence of these categories (PC1 $\approx$ 47%). The full 25-category analysis remains the primary specification, with the vice-excluded version providing a complementary check on the robustness of the findings.

## 6. Per-Era Distribution & Era Comparison

In [ ]:
# Inspect CLR dict structure
clr_dict = epsilon_sweep_result['clr_dict']
chosen_epsilon = chosen_eps

# Top-Level Diagnostics
print(f"\n📁 CLR DICTIONARY METADATA")
print("═" * 120)
print(f"Type:         {type(clr_dict)}")
print(f"Total Keys:   {len(clr_dict)}")

# Format keys to 6 decimal places for display
formatted_keys = [f"{float(k):.15f}" for k in list(clr_dict.keys())[:5]]
print(f"Keys (Head):  {formatted_keys} ...")


# Targeted Epsilon Drill-down
if chosen_epsilon in clr_dict:
    chosen_clr = clr_dict[chosen_epsilon]
    print(f"\n🔍 DRILL-DOWN: EPSILON = {chosen_epsilon:.15f}")
    print("═" * 120)
    print(f"Matrix Type:   {type(chosen_clr)}")
    print(f"Matrix Shape:  {chosen_clr.shape}")
    print(f"Total Columns: {len(chosen_clr.columns)}")
    
    # Format DatetimeIndex to YYYY-MM
    formatted_index = chosen_clr.index[:5].strftime('%Y-%m').tolist()
    print(f"Index (Head):  {formatted_index} ...")
    print(f"Columns (Head):{chosen_clr.columns[:5].tolist()} ...")
else:
    print(f"[ERROR] Epsilon {chosen_epsilon:.15f} is missing from the dictionary keys.")

### <a id="6.1-Slice-Into-Eras"></a> Build the CLR Matrix & Slice Into Eras

In [ ]:
# Build CLR Matrix into Eras
clr_era = clr.slice_clr_into_eras(epsilon_sweep_result['clr_dict'], cfg.era_boundaries, eps=chosen_eps)

### <a id="6.2-Per-Era-Parameters"></a> Compute Per-Era Distributional Parameters

In [ ]:
# Compute Per-Era Distribution
eras_stat = clr.compute_era_distribution_parameters(clr_era, verbose=True)

#### 📌 Summary

> - The per-era analysis shows that Chicago's crime composition changed substantially between the pre-COVID, COVID, and post-COVID periods. The largest and most permanent shifts involved vice-related offenses. Gambling and Prostitution declined sharply during COVID and remained far below their pre-pandemic levels afterward, making them the strongest structural changes in the dataset. At the same time, violent and weapons-related offenses became relatively more prominent during the COVID era, including increases in Weapons Violations, Homicide, and Aggravated Assault. Some of these increases eased after COVID, but they did not fully return to earlier patterns. Motor Vehicle Theft showed the clearest long-term increase, rising steadily across all three periods without reversing. Fraud also increased during COVID, while Drug Abuse Violations declined.
> - The analysis also found that the COVID period was much more unstable and volatile than the years before or after it, especially for lower-frequency categories such as Gambling, Prostitution, Embezzlement, and Stolen Property. Most of this instability settled down after COVID, although Embezzlement remained unusually volatile compared with earlier years. Importantly, the study measures changes in the relative composition of crime categories rather than raw crime totals, meaning the results describe how the balance of offenses shifted within the overall crime system.
> - Overall, the findings point to three characteristically different periods: a relatively stable pre-COVID period, a sharp COVID-era disruption, and a post-COVID period that settled into a new structure rather than returning to the old one.

### <a id="6.3-Covariance-Sanity-Check"></a> Covariance Sanity Check
`era_covs_lw`: LW takes your noisy estimated relationships between crime types and asks, "How much of this is real signal vs. sampling noise?" It preserves the signal and reduces noise toward zero.

In [ ]:
# Configuration for Audit Report (Sanity Check)
audit_config = {
    'era_covs': "Covariance Matrix Geometry",
    'era_means': "Era Mean Matrix Structure",
    'era_stds': "Era Std Deviation Matrix Structure"
}

width = 65
row_fmt = "  {:<35} | {:>22}"
divider = "─" * width

print("═" * width)
print(f"{'📋 DATA METRIC AUDIT REPORT':^{width}}")
print(f"{'Structural Dimensionality & Sanity Check':^{width}}")
print("═" * width)
print(row_fmt.format("Report Target / Node", "Dimensions"))
print(divider)

for key, title in audit_config.items():
    data = eras_stat.get(key)
    if data is None:
        continue

    # Section Header
    print(f"\n🔹 {title.upper()}")

    # Case A: Nested Era-Specific Covariance Dictionaries
    if isinstance(data, dict):
        items = list(data.items())
        for idx, (era, matrix) in enumerate(items):
            branch = "└── " if idx == len(items) - 1 else "├── "
            dim_str = f"{matrix.shape[0]} × {matrix.shape[1]} (Square)"
            print(row_fmt.format(f"{branch}{era}", dim_str))

    # Case B: Integrated DataFrames
    elif hasattr(data, 'shape'):
        rows, cols = data.shape
        print(row_fmt.format("├── Integrated Dataset", f"{rows} × {cols}"))
        print(row_fmt.format("├── Analyzed Features", f"{rows} Crime Categories"))
        print(row_fmt.format("└── Temporal Breakdown", "3 Eras + 3 Shift Metrics"))

    print(divider)

### <a id="6.4-Heatmaps"></a> Visualize the Mean & Volatility CLR Shift Heatmap

In [ ]:
# Heat Map for Mean
plot.plot_era_heatmaps(eras_stat, stat_type='mean')

In [ ]:
# Heat Map for Volatility
plot.plot_era_heatmaps(eras_stat, stat_type='std')

#### 📌 Summary

> - The mean and volatility heatmaps show that the COVID-era disruption was not evenly spread across all crime categories. Instead, the largest compositional changes were concentrated in four areas: Gambling, Prostitution, Embezzlement, and Stolen Property. These categories experienced much larger shifts than the rest of the system, creating a clear separation between the most disrupted offenses and all others. 
> - The volatility results also show that much of the instability observed during COVID later subsided, especially for Prostitution and Stolen Property, which moved closer to their pre-COVID behavior after the pandemic. Embezzlement, however, remained unusually volatile even after COVID, making it a notable exception.
> - The mean-shift results further show that the post-COVID period did not simply return to pre-pandemic conditions. Property-related offenses, especially Motor Vehicle Theft and Stolen Property, became relatively more prominent after COVID, while some of the sharp COVID-era increases in vice and weapons-related categories partially reversed. Together, these patterns suggest that Chicago entered a new post-COVID crime composition rather than returning to its earlier structure.

### <a id="6.5-Distribution-Audit"></a> Build & Check the CLR Matrix & Slice Into Eras

In [ ]:
# Combined structural audit of CLR‑era matrices and the reconstructed wide‑format dataset
clr_df, raw_wide = utils.audit_clr_and_wide(clr_era = clr_era, fill_results=fill_results)

#### Per-Category CLR Structural Distribution Plots

In [ ]:
# Perform an optimized per‑category distribution audit on a CLR‑transformed dataset
diag_result = plot.check_clr_distributions(clr_df=clr_df, raw_counts=fill_results['filled_df'], epsilon=chosen_eps)

In [ ]:
# The diagnostics table, worst-behaved categories first
print(diag_result.to_string(index=False))

# Just the flagged ones
flagged = diag_result[diag_result['flagged']]
print(f"\nFlagged categories ({len(flagged)}):")
print(flagged[['category', 'skew', 'excess_kurtosis', 'zero_rate_%']].to_string(index=False))

#### 📌 Summary

> The CLR distribution diagnostics at $\varepsilon = 0.008938$ show that the transformation performs well across nearly the entire dataset. Most crime categories (21 out of 25) display low skew and approximately normal distributions after transformation, indicating that the data are stable and suitable for compositional analysis. Only four lower-volume categories, Stolen Property, Embezzlement, Prostitution, and Gambling, show heavier tails and more uneven distributions. Importantly, the results suggest that these irregularities are not caused by the pseudocount itself. In fact, the sparsest category, Gambling, has relatively mild tail behavior, while Stolen Property, which rarely contains zeros, shows the strongest extremes. This pattern indicates that the heavy tails are driven by real-world spikes and structural changes in the data rather than mathematical distortion from the transformation. Overall, the findings support the conclusion that the selected ε value is appropriate and does not artificially distort even the sparse categories in the analysis.

## 6a & 6b: Stationarity Testing & Block Bootstrap on the CLR series 
The CLR-transformed series spans 304 months (Jan 2001 - April 2026) across three eras defined by administrative and policy boundaries: the pre-COVID period, the COVID disaster period (Mar 2020 - Dec 2022), and the post-COVID period. Over this span, Chicago crime composition was subject to major policy shifts (changes in gambling enforcement, the 2019 CPD consent decree), demographic change, and a global pandemic, all of which could introduce sustained distributional shifts inconsistent with stationarity. Non-stationarity is therefore theoretically expected and consistent with the structural break detection framework adopted here. We nonetheless conduct formal stationarity testing to characterize each series and to distinguish genuine breaks from gradual trends.

**Note:**
> Standard ADF and KPSS tests assume no structural breaks and independent observations. Given the imposed era boundaries (months 230 and 264), the compositional constraint across K = 25 series, and sparse categories with imputed zeros, results are interpreted as indicative rather than definitive, and conflicting ADF/KPSS outcomes are reported transparently. To assess the significance of era-to-era shifts while respecting temporal autocorrelation, a block bootstrap resamples contiguous month blocks, preserving the serial dependence that a standard bootstrap would break.

### <a id="6a-Stationarity"></a> 6a: Stationarity Testing (ADF / KPSS / Zivot-Andrews)

In [ ]:
# Stationarity - CLR
stat_df = clr.run_stationarity_analysis(clr_df, fill_results['filled_df'], era_boundaries=cfg.era_boundaries, 
                                       seasonal_cycle=12, sparse_threshold=0.05, verbose=True)

In [ ]:
# Initialize values
window  = 12
category = 'Arson'
row = stat_df.loc[category, ['za_date', 'za_stat', 'za_p_adj']]
za_date, za_stat, za_p_adj = row
# Plot
arson_plot = plot.plot_structural_break(
        clr_df, 
        category, 
        break_date=za_date,
        window=window,
        za_stat=za_stat, 
        za_p_adj=za_p_adj, 
        era_boundaries=cfg.era_boundaries
)

In [ ]:
# Initialize values
category = 'Prostitution'
row = stat_df.loc[category, ['za_date', 'za_stat', 'za_p_adj']]
za_date, za_stat, za_p_adj = row
# Plot
gambling_plot = plot.plot_structural_break(
        clr_df, 
        category, 
        break_date=za_date,
        window=window,
        za_stat=za_stat, 
        za_p_adj=za_p_adj, 
        era_boundaries=cfg.era_boundaries
)

In [ ]:
# Initialize values
category = 'Liquor Laws'
row = stat_df.loc[category, ['za_date', 'za_stat', 'za_p_adj']]
za_date, za_stat, za_p_adj = row
# Plot
gambling_plot = plot.plot_structural_break(
        clr_df, 
        category, 
        break_date=za_date,
        window=window,
        za_stat=za_stat, 
        za_p_adj=za_p_adj, 
        era_boundaries=cfg.era_boundaries
)

In [ ]:
# Classify each dense category by ZA significance (using ADJUSTED p)
summary = stat_df[~stat_df['is_sparse']].copy()
summary['za_significant'] = summary['za_p_adj'] < 0.05

sig = summary[summary['za_significant']].sort_values('za_p_adj')
print(f"Significant structural breaks (ZA, FDR-adjusted, α=0.05): {len(sig)} of {len(summary)}")
print(sig[['za_date', 'za_stat', 'za_p', 'za_p_adj']].to_string())

print(f"\nNon-significant (no single clean break): {(~summary['za_significant']).sum()}")

#### 📌 Summary

> - The stationarity and structural-break analysis show that Chicago's crime composition changed over time, but not all categories changed in the same way or at the same time. Most categories were non-stationary, meaning their relative shares evolved gradually across the 2001–2026 period rather than remaining stable. Formal Zivot-Andrews testing, after correcting for multiple comparisons, identified only three categories with statistically significant single structural breaks: Arson, Prostitution, and Liquor Laws. Prostitution experienced the largest confirmed shift, dropping sharply during the COVID period and remaining at a much lower level afterward, while Arson stepped up to a new, stable level in early 2020. Liquor Laws showed a significant break much earlier, in 2015, demonstrating that important structural changes were already underway well before the pandemic.
> - By contrast, categories such as Weapons Violations and Homicide displayed strong COVID-era movement visually, yet the statistical tests indicated these changes were better explained by gradual trends or temporary spikes than by permanent regime shifts.
> - Overall, the findings suggest that Chicago's crime composition evolved mainly through long-term gradual change, punctuated by only a small number of true structural breaks. COVID clearly contributed an important shock to the system, most notably for Prostitution, but it was not the sole driver of change. The results also underscore the value of formal statistical testing: several categories that appeared to "break" on visual inspection were, in fact, trending over many years or later partially reverted. Together, the analyses support a more nuanced interpretation than a single-moment disruption: Chicago's crime structure was already changing before 2020, and the pandemic intensified or accelerated some of those ongoing shifts rather than resetting the system at one point in time.

### <a id="6b-Bootstrap"></a> 6b: Block Bootstrap Test

In [ ]:
# Bootstrap Comparison of Eras
sparse_cats = ['Gambling']

# Pre-COVID vs COVID
bootstrap_pc = clr.run_era_comparison(
    clr_era['Pre-COVID'], clr_era['COVID'],
    label1='Pre-COVID', label2='COVID',
    sparse_cats=sparse_cats,
    block_size=6,
    seed=_RANDOM_STATE, verbose=True
)

In [ ]:
# COVID vs Post-COVID
bootstrap_cp = clr.run_era_comparison(
    clr_era['COVID'], clr_era['Post-COVID'],
    label1='COVID', label2='Post-COVID',
    sparse_cats=sparse_cats,
    block_size=6,
    seed=_RANDOM_STATE, verbose=True
)

In [ ]:
# Pre-COVID vs Post-COVID
bootstrap_pp = clr.run_era_comparison(
    clr_era['Pre-COVID'], clr_era['Post-COVID'],
    label1='Pre-COVID', label2='Post-COVID',
    sparse_cats=sparse_cats,
    block_size=6,
    seed=_RANDOM_STATE, verbose=True
)

#### 📌 Summary

> - The bootstrap quantifies how Chicago's crime composition differs across the three pre-defined eras. After flagging Gambling as a sparse category and excluding it from the multiple-testing correction, the comparison between the pre-COVID and COVID periods returns statistically significant mean differences across all 24 dense crime categories, consistent with a broad COVID-era compositional shift, though the asymmetric window lengths (230 vs. 34 months) mean part of this signal may reflect long-running drift rather than a discrete shock. The largest compositional shifts occurred in violent and weapons-related offenses, while Drug Abuse Violations showed a major decline.
> - From COVID to Post-COVID, a comparison of two windows of similar length shows that only Homicide, Weapons Violations, and Burglary remain significant after FDR correction, indicating that the system largely stabilized rather than continuing to reorganize. The effect-size and significance gap (e.g., Motor Vehicle Theft, Arson) underscores the importance of reading both columns together rather than relying on FDR-adjusted p-values alone.
> - The Pre-COVID to Post-COVID comparison shows 21 of 24 categories significantly different, consistent with persistent compositional change, though the same window asymmetry (230 vs. 40 months) applies.
> - These results describe differences across periods defined in advance by policy and pandemic milestones, but they do not establish that those boundaries mark the underlying compositional shifts. The uneven window sizes (230 vs. 34 vs. 40 months), the ZA finding of an earlier Liquor Laws break in 2015, and the gradual pre-COVID trajectory of PC1 together suggest that some of the differences detected across the Pre-COVID seam may reflect long-term drift rather than a distinct pandemic shock. To separate gradual evolution from structural breaks, the next section applies multivariate, data-driven changepoint detection to the CLR series.

## 6c-1 - 6c.4: Multivariate Changepoint Detection

### <a id="6c.1-Dynp-Confirmatory"></a> 6c.1: Confirmatory: Dynp at n_bkps=2

In [ ]:
# Assuming your dataframe is named clr_df (sanity check)
row_sums = clr_df.sum(axis=1)

# Check if the maximum absolute deviation from zero is within floating-point tolerance
is_zero_sum = np.allclose(row_sums, 0, atol=1e-12)

print("--- Zero-Sum Validation ---")
print(f"Shape: {clr_df.shape}")
print(f"Max Row Sum Deviation: {row_sums.abs().max():.2e}")
print(f"Is True Zero-Sum?     : {is_zero_sum}")

In [ ]:
# Assuming your dataframe is named clr_df (sanity check)
row_sums = clr_matrix.sum(axis=1)

# Check if the maximum absolute deviation from zero is within floating-point tolerance
is_zero_sum = np.allclose(row_sums, 0, atol=1e-12)

print("--- Zero-Sum Validation ---")
print(f"Shape: {clr_matrix.shape}")
print(f"Max Row Sum Deviation: {row_sums.abs().max():.2e}")
print(f"Is True Zero-Sum?     : {is_zero_sum}")

In [ ]:
# Extract dimensions and standardize inputs
X = clr_df.values                         # Shape: (304, 25)
dates = clr_df.index
MIN_SIZE = 24                             # Strict 2-year macro-regime constraint

# Execute Dynamic Programming Search with RBF Kernel
algo = rpt.Dynp(model='rbf', min_size=MIN_SIZE).fit(X)
bkps = algo.predict(n_bkps=2)

# Cleanly map breakpoint indices back to dataframe timestamps
# Note: ruptures includes the final index len(X) as the last element in bkps.
# We slice [:-1] to isolate just the true internal transition breaks.
detected = [dates[i - 1] for i in bkps[:-1]]

# Explicitly isolate your target baseline timestamps
assumed_target_1 = dates[229]
assumed_target_2 = dates[263]

# Compute calendar-exact month gaps using pandas Period arithmetic
# This bypasses variable day-count drift (30 vs 31 days) over long timelines.
period_detected_0 = pd.Period(detected[0], freq='M')
period_detected_1 = pd.Period(detected[1], freq='M')

period_target_1 = pd.Period(assumed_target_1, freq='M')
period_target_2 = pd.Period(assumed_target_2, freq='M')

gap_1_months = abs((period_detected_0 - period_target_1).n)
gap_2_months = abs((period_detected_1 - period_target_2).n)

# Print a structured pipeline evaluation report
print("==================================================")
print("     REGIME CHANGE-POINT DETECTION REPORT         ")
print("==================================================")
print(f"Algorithmic Breaks (RBF) : {[d.strftime('%Y-%m') for d in detected]}")
print(f"Historical Assumed Breaks: {assumed_target_1.strftime('%Y-%m')}, {assumed_target_2.strftime('%Y-%m')}")
print("--------------------------------------------------")
print(f"Gap to Target Break 1    : {gap_1_months} months")
print(f"Gap to Target Break 2    : {gap_2_months} months")
print("==================================================")

### <a id="6c.2-Sensitivity"></a> 6c.2: Sensitivity: Vice-Included vs Vice-Excluded

In [ ]:
# Version A: unscaled CLR with all 25 categories
X_A = clr_df.values

# Version C: unscaled CLR with vice categories removed (23 categories)
X_C = clr_matrix.values

# Explicitly define historical targets to ensure headers match the math
target_1_dt = pd.Timestamp('2020-02-01')
target_2_dt = pd.Timestamp('2022-12-01')

target_1_per = pd.Period(target_1_dt, freq='M')
target_2_per = pd.Period(target_2_dt, freq='M')

# Run identical changepoint detection configurations on both versions
results = []
for name, X in [
    (f'A: unscaled CLR, {X_A.shape[1]} cats (primary)', X_A),
    (f'C: unscaled CLR, {X_C.shape[1]} cats (no vice)',  X_C)
]:
    # Execute global dynamic programming segmentation with RBF Kernel
    algo = rpt.Dynp(model='rbf', min_size=MIN_SIZE).fit(X)
    bkps = algo.predict(n_bkps=2)
    
    # Map index boundaries to true timestamps (ignoring right-exclusive end step)
    detected = [dates[i - 1] for i in bkps[:-1]]
    
    # Convert detected breaks to calendar periods
    break_1_per = pd.Period(detected[0], freq='M')
    break_2_per = pd.Period(detected[1], freq='M')
    
    # Compute calendar-exact month differences (bypasses 30 vs 31 day drift)
    gap_1 = abs((break_1_per - target_1_per).n)
    gap_2 = abs((break_2_per - target_2_per).n)

    results.append({
        'version': name,
        'break_1': detected[0].strftime('%Y-%m'),
        'break_2': detected[1].strftime('%Y-%m'),
        f'gap_to_{target_1_per}': f'{gap_1}m',
        f'gap_to_{target_2_per}': f'{gap_2}m'
    })

# Print verified comparison table
print("=" * 95)
print(f"{'CHANGEPOINT SENSITIVITY: Vice-included vs Vice-excluded':^95}")
print("=" * 95)
print(pd.DataFrame(results).to_string(index=False))
print("=" * 95)

#### 📌 Summary

Following the convention adopted in the PCA sensitivity analysis (Section 5), $\varepsilon$ is held fixed at the 25-category optimum ($\varepsilon = 0.00894$) across both specifications. The 23-category sub-composition was previously verified to remain numerically stable at this pseudocount (max|CLR| = 10.12, within the stable region of the epsilon sweep). Holding $\varepsilon$ fixed isolates the effect of vice removal from any effect of pseudocount choice.

### <a id="6c.3-Pelt"></a> 6c.3: Pelt Exploratory Penalty Sweep

In [ ]:
# Tests how many regimes the data supports without fixing n_bkps a priori.
# BIC penalty (log(n) × d) is scaled per-version to keep the relative penalty
# comparable across dimensionalities (A=25 cats, C=23 cats).

# Timeline length is shared; dimensionality differs per specification
n      = X_A.shape[0]
_, d_A = X_A.shape
_, d_C = X_C.shape

# BIC baseline: log(n) × d. Each version uses its own d to scale correctly
# with the cost reductions in its own dimensionality.
base_pen_A = np.log(n) * d_A
base_pen_C = np.log(n) * d_C

# Multipliers span AIC-permissive (0.25× BIC) to strongly conservative (4× BIC)
multipliers = [0.25, 0.5, 1.0, 2.0, 4.0]

# Run the sweep across both specifications
rows = []
for label, X, base_pen in [
    (f'A: {d_A}-cat (primary)', X_A, base_pen_A),
    (f'C: {d_C}-cat (no vice)',  X_C, base_pen_C),
]:
    # PELT segmentation with RBF kernel; sensitive to mean and covariance shifts
    algo = rpt.Pelt(model='rbf', min_size=MIN_SIZE).fit(X)

    for mult in multipliers:
        # Final penalty value for this iteration
        penalty = mult * base_pen
        
        # Detect breakpoints; ruptures appends the final index, so slice [:-1]
        bkps = algo.predict(pen=penalty)
        detected = [dates[i - 1] for i in bkps[:-1]]
        
        # Format date list for display (or sentinel if no breaks detected)
        date_str = ', '.join(d.strftime('%Y-%m') for d in detected) if detected else '(none)'
        
        rows.append({
            'Specification':  label,
            'Penalty Mult':   mult,
            'Penalty Value':  round(penalty, 2),
            'Num Breaks':     len(detected),
            'Detected Dates': date_str,
        })

sweep_df = pd.DataFrame(rows)

# Print structured comparison table
print("=" * 110)
print(f"{'PELT EXPLORATORY PENALTY SWEEP (RBF KERNEL)':^110}")
print("=" * 110)
print(sweep_df.to_string(index=False))
print("=" * 110)

# Stability summary: which break count is most consistent across the sweep
print(f"\n{'STABILITY SUMMARY':^110}")
print("-" * 110)
for spec in sweep_df['Specification'].unique():
    sub = sweep_df[sweep_df['Specification'] == spec]
    counts = sub['Num Breaks'].value_counts()
    modal_n = counts.idxmax()
    n_at_modal = counts.max()
    print(f"  {spec:35s} | most stable at {modal_n} breaks "
          f"({n_at_modal} of {len(sub)} penalty values)")
print("-" * 110)

#### 📌 Summary

Exploratory PELT changepoint analysis found no breakpoints under standard BIC-level penalties in either the 25-category or 23-category crime composition models, and this remained true across penalty multipliers from 0.5× to 4× BIC. Only under a very permissive penalty (0.25× BIC) did a single weak breakpoint appear, centered around mid-2015 in both specifications. This timing loosely aligns with earlier findings involving Liquor Laws, Gambling decline, and Prostitution transitions, but the signal was too weak to survive standard model-selection penalties.

The confirmatory Dynp analysis, which was forced to identify two breakpoints, returned dates at 2013-11 and 2019-09 for the full model and 2008-06 and 2015-12 for the vice-excluded model. However, because Dynp always returns the requested number of breaks, these results indicate only where breaks are placed when imposed, not proof that true regime shifts exist. The lack of support from PELT suggests the improvement in fit was too small to justify the added complexity.

Overall, the changepoint analysis does not support discrete structural regime shifts in Chicago crime composition from 2001 to 2026, including around COVID. Instead, the evidence points to gradual compositional evolution over time, consistent with the earlier stationarity findings. The policy-defined eras used elsewhere in the study remain useful for descriptive comparison, but should not be interpreted as empirically validated regime boundaries.

In [ ]:
print("Pelt min_size sensitivity (Version A, BIC penalty):")
for ms in [12, 18, 24, 36]:
    algo = rpt.Pelt(model='rbf', min_size=ms).fit(X_A)
    b = algo.predict(pen=base_pen_A)
    dates_found = [dates[i-1].strftime('%Y-%m') for i in b[:-1]]
    print(f"  min_size={ms:>2}m -> {len(dates_found)} breaks: {dates_found if dates_found else '(none)'}")

### <a id="6c.4-Visualization"></a> 6c.4: PC1 Trajectory Visualization

In [ ]:
# Plot Regime Segmentation
plot_result = plot.plot_regime_segmentation(
    dates=clr_df.index, 
    pca_coords=pca_A['coords_norm'][:, 0], 
    breaks_A=['2013-11', '2019-09'], 
    breaks_C=['2008-06', '2015-12'],
    pelt_break='2015-07',
)


## Section 7: Per-Era Co-Movement Structure

### <a id="7a-Frobenius"></a> 7a Frobenius Distance Between Era Correlation Matrices

####  7a - Per-era co-movement structure: does the dependence pattern between crime categories itself shift across eras, or do only the means change?
> We split the Pre-COVID era (the longest, 230 months) in half and measure the within-era Frobenius distance between the two halves.
> We convert each era's Ledoit-Wolf-shrunk covariance matrix to a correlation matrix, then measure the Frobenius distance between the correlation matrices of each pair of eras. Frobenius distance is the natural matrix norm here: it sums the squared differences between corresponding entries, so it captures overall divergence in dependence structure.
> Interpretation requires a noise baseline. We split the Pre-COVID era (the longest, 230 months) in half and measure the within-era Frobenius distance between the two halves. This quantifies the distance attributable solely to sample variation under a stable structure. Cross-era distances substantially larger than this baseline indicate genuine structural change.

In [ ]:
def cov_to_corr(cov_matrix):
    """Convert covariance to correlation via diagonal-scaling broadcast."""
    d = np.sqrt(np.diag(cov_matrix))                 # standard deviations from diagonal
    return cov_matrix / d[:, None] / d[None, :]      # normalize to correlation matrix

def frobenius_dist(A, B):
    """Frobenius norm of the difference between two matrices."""
    return np.linalg.norm(A - B, ord='fro')          # sqrt(sum of squared differences)

# Per-era correlation matrices (convert LW covariances to correlations)
corr = {era: cov_to_corr(C) for era, C in eras_stat['era_covs_lw'].items()}

# Cross-era Frobenius distances for selected era pairs
pairs = [('Pre-COVID', 'COVID'),
         ('COVID',     'Post-COVID'),
         ('Pre-COVID', 'Post-COVID')]
cross_distances = {f"{a} ↔ {b}": frobenius_dist(corr[a], corr[b]) for a, b in pairs}

# Baseline 1: split Pre-COVID into two halves and compute distance between them
pre_clr_vals = clr_era['Pre-COVID'].values          # full Pre-COVID CLR matrix
half_idx = len(pre_clr_vals) // 2                   # midpoint index
lw_h1 = LedoitWolf().fit(pre_clr_vals[:half_idx])   # LW covariance on first half
lw_h2 = LedoitWolf().fit(pre_clr_vals[half_idx:])   # LW covariance on second half
baseline_halves = frobenius_dist(                   # distance between the two halves
    cov_to_corr(lw_h1.covariance_),
    cov_to_corr(lw_h2.covariance_)
)

# Baseline 2: short-window resampling to match COVID sample size (34 months)
rng = np.random.default_rng(_RANDOM_STATE)          # reproducible RNG
window_size = 34                                    # match COVID sample length
within_short = []                                   # store distances for resampled windows
for _ in range(50):                                 # 50 random window pairs
    starts = rng.choice(len(pre_clr_vals) - window_size, size=2, replace=False)
    w1 = pre_clr_vals[starts[0]:starts[0] + window_size]   # first random window
    w2 = pre_clr_vals[starts[1]:starts[1] + window_size]   # second random window
    c1 = LedoitWolf().fit(w1).covariance_                   # LW covariance for window 1
    c2 = LedoitWolf().fit(w2).covariance_                   # LW covariance for window 2
    within_short.append(frobenius_dist(                     # distance between windows
        cov_to_corr(c1), cov_to_corr(c2)
    ))
baseline_short_mean = np.mean(within_short)                 # average within-era distance
baseline_short_p95 = np.percentile(within_short, 95)        # 95th percentile threshold

# Pretty console report formatting
WIDTH = 90
LINE = "=" * WIDTH
SUB = "-" * WIDTH

print(LINE)
print(f"{'PER-ERA CO-MOVEMENT STRUCTURE COMPARISON':^{WIDTH}}")   # centered title
print(f"{'Frobenius distance between correlation matrices':^{WIDTH}}")
print(LINE)

# Header row for comparison table
print(f"{'Comparison':<28} {'Frobenius':>12} {'vs. halves':>14} {'vs. short':>14} {'Flag':>6}")
print(SUB)

# Main rows: cross-era distances with ratios and flag if above 95th percentile
for label, dist in cross_distances.items():
    ratio_halves = dist / baseline_halves                    # relative to halves baseline
    ratio_short  = dist / baseline_short_mean                # relative to short-window mean
    flag = "★" if dist > baseline_short_p95 else ""          # mark if above 95th percentile
    print(
        f"{label:<28}"
        f"{dist:>12.3f}"
        f"{ratio_halves:>14.2f}×"
        f"{ratio_short:>14.2f}×"
        f"{flag:>6}"
    )

print(SUB)

# Baseline rows
print(f"{'Baseline: Pre-COVID halves':<28}{baseline_halves:>12.3f}{'1.00×':>15}")
print(f"{'Baseline: short windows (mean)':<28}{baseline_short_mean:>10.3f}{'1.00×':>30}")
print(f"{'  (95th percentile)':<28}{baseline_short_p95:>12.3f}")
print(LINE)

# Interpretation block
print("\nINTERPRETATION")
print("-" * 20)
print("• Two baselines are used:")
print("    1) Pre-COVID split in halves (~115 months each), matching Pre/Post sample sizes.")
print("    2) Fifty pairs of randomly-sampled 34-month windows, matching COVID sample size.\n")
print("• The 95th percentile of the short-window distribution is the threshold a")
print("  cross-era distance must exceed to indicate genuine structural change.\n")
print("• Rows marked with ★ exceed this threshold.")


#### 📌 Summary

Beyond shifts in mean shares (Section 6b), we test whether the *dependence structure* between crime categories itself changed across eras. We convert each era's Ledoit-Wolf-shrunk covariance matrix into a correlation matrix and measure the Frobenius distance between each pair of eras. Two within-era baselines anchor interpretation: 
1. Pre-COVID split into 115-month halves with independent LW fits (matching Pre/Post-COVID sample sizes)
2. 50 pairs of 34-month windows resampled from Pre-COVID (matching COVID sample size).

All three cross-era Frobenius distances substantially exceed the 95th percentile of the short-window baseline (5.10): Pre↔COVID = 8.11 (1.99× baseline mean), COVID↔Post-COVID = 5.74 (1.41×), Pre↔Post-COVID = 9.14 (2.24×). Each cross-era comparison sits in the upper tail of the distribution of within-Pre-COVID short-window distances, indicating that the dependence structure between crime categories shifts not only with their mean shares but also across each era boundary.

The halves baseline (8.65) is larger than two of the three cross-era distances. This baseline absorbs nineteen years of within-Pre-COVID gradual drift (2001–2020 spans the entire pre-pandemic panel), and is therefore too lenient as a reference for shorter cross-era comparisons. The short-window baseline, which dilutes long-run drift through repeated random sampling, is the appropriate reference.

The Pre <-> Post-COVID distance (9.14) exceeds both the Pre <-> COVID distance (8.11) and the COVID <-> Post-COVID distance (5.74), indicating that the post-COVID dependence structure did not revert toward the pre-pandemic pattern after the pandemic subsided. The system's co-movement structure continued evolving in the same direction. This corroborates, on a second-moment dimension, the bootstrap finding (Section 6b) that 21 of 24 categories remained significantly different between Pre-COVID and Post-COVID, and is consistent with the descending trajectory of PC1 scores (Section 5).

Characterizing which specific category-pairs drove the cross-era dependence shifts requires per-era PCA decompositions, which the COVID-era sample size (34 months) does not reliably support. The Frobenius summary answers whether the structure changed without overcommitting to specific category-pair reorganizations; the next section uses element-wise correlation differences to identify the largest specific shifts at a less demanding statistical bar.

### <a id="7b-Element-Wise"></a> 7b Crime-Pair Correlation Shifts (Element-Wise)
> Which crime-pair correlations shifted most across eras?

In [ ]:
# Element-wise differences between per-era correlation matrices.
categories = list(corr['Pre-COVID'].columns) if hasattr(corr['Pre-COVID'], 'columns') \
             else [f'cat_{i}' for i in range(corr['Pre-COVID'].shape[0])]   # category names fallback

# Convert correlation matrices to numpy arrays for fast indexing
corr_arrays = {era: np.asarray(C) for era, C in corr.items()}

def top_changes(C1, C2, label1, label2, top_n=10):
    """Find and display the category-pairs whose correlation changed most between two eras."""

    diff = C2 - C1                                      # elementwise correlation differences

    n = diff.shape[0]                                   # number of categories
    iu, ju = np.triu_indices(n, k=1)                    # upper triangle indices (skip diagonal)

    # Build list of (category_i, category_j, corr_in_era1, corr_in_era2, delta)
    pairs = [
        (categories[i], categories[j], C1[i, j], C2[i, j], diff[i, j])
        for i, j in zip(iu, ju)
    ]

    pairs.sort(key=lambda x: abs(x[4]), reverse=True)   # sort by absolute change magnitude

    print(f"\nTop {top_n} correlation shifts: {label1} → {label2}")   # header
    print("─" * 80)
    print(f"{'Pair':<50}{label1:>10}{label2:>10}{'Δ':>10}")           # column titles
    print("─" * 80)

    # Print top N category-pair shifts
    for cat_i, cat_j, c1, c2, d in pairs[:top_n]:
        pair_label = f"{cat_i[:24]} ↔ {cat_j[:24]}"     # truncate long names for alignment
        print(f"{pair_label:<50}{c1:>10.2f}{c2:>10.2f}{d:>+10.2f}")

    print("─" * 80)

# Run comparisons for Pre vs Post and Pre vs COVID
top_changes(corr_arrays['Pre-COVID'], corr_arrays['Post-COVID'],
            'Pre', 'Post', top_n=10)

top_changes(corr_arrays['Pre-COVID'], corr_arrays['COVID'],
            'Pre', 'COVID', top_n=10)


#### 📌 Summary

The Frobenius distance analysis showed that the dependence structure between crime categories shifted across all era boundaries. Element-wise differences between the Pre-COVID and Post-COVID correlation matrices identify the specific pairs driving these shifts.

The ten largest absolute correlation changes are concentrated in two categories: **Drug Abuse Violations** (appearing in 5 of the top 10 Pre→Post shifts) and **Liquor Laws** (appearing in 5 of the top 10 shifts). Both categories show the same pattern: strong negative correlations with most other categories in Pre-COVID, then swinging to near-zero or positive correlations in Post-COVID. Examples include Drug Abuse Violations <-> Fraud (−0.59 → +0.44), Drug Abuse Violations <-> Aggravated Assault (−0.69 -> +0.18), and Liquor Laws <-> Fraud (−0.78 -> +0.35).

This is best read as a compositional-mechanics finding rather than a substantive realignment of crime types. In CLR space, when a single category occupies a large share of the composition, its month-to-month share movements mechanically push every other category's share in the opposite direction, producing systemic negative correlations. Drug Abuse Violations had a pre-COVID mean share corresponding to CLR ≈ 2.13, the second-largest of any category in the panel. Its sharp decline (Pre→Post shift of −1.03 in CLR units, documented in Section 6b) reduced its dominance of the composition. The negative correlations it imposed on other categories weakened or reversed as a mechanical consequence.

The Pre→COVID shifts show the same pattern, with Drug Abuse Violations again appearing in 5 of the top 10 and the vice categories (Gambling, Prostitution) contributing additional dependence shifts. The Pre→Post result is the cleaner read because both windows have sufficient observations to yield stable correlation estimates; the Pre <-> COVID comparison is reported for completeness but should be interpreted with the COVID-era 34-month sample size in mind.

This finding is consistent with the broader thesis of the paper: the post-pandemic dependence structure differs from pre-pandemic patterns primarily because the categories that dominated the pre-pandemic composition (Drug Abuse Violations, Liquor Laws, vice) shrank, releasing their compositional pressure on the rest of the system, rather than because new co-movement clusters formed among the remaining categories.